# Transfer Learning — Reusing Pretrained Models

## Goal
Instead of training from scratch, reuse a model already trained on millions of images.
Fine-tune only the last layer for our specific task.

## Why Transfer Learning?
Training ResNet from scratch on ImageNet takes weeks on GPUs.
Fine-tuning takes minutes on a laptop.

## What We Build
Use ResNet-18 pretrained on ImageNet.
Replace the final layer for a new classification task.
Freeze all layers except the last one — train only the classifier.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {device}")

Device: mps


## 1. Load Dataset with ImageNet Preprocessing
ResNet expects 224x224 images normalized with ImageNet statistics.

In [2]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

# ImageNet normalization — required for pretrained models
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),  # MNIST is grayscale, ResNet needs 3 channels
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

# Use a subset of MNIST for speed
full_dataset = datasets.MNIST(
    root="../data/raw",
    train=True,
    download=True,
    transform=transform
)

# Use only 2000 samples — transfer learning needs less data
subset_size = 2000
subset, _ = random_split(full_dataset, [subset_size, len(full_dataset) - subset_size])

train_size = int(0.8 * subset_size)
val_size = subset_size - train_size
train_dataset, val_dataset = random_split(subset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

Training samples: 1600
Validation samples: 400


## 2. Load Pretrained ResNet-18 and Modify for MNIST

In [3]:
# Load pretrained ResNet-18
model = models.resnet18(weights='IMAGENET1K_V1')

# Freeze all layers
for param in model.parameters():
    param.requires_grad = False

# Replace final layer — MNIST has 10 classes
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 10)

# Only final layer is trainable
model = model.to(device)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters: {total_params - trainable_params:,}")
print(f"\nOnly {trainable_params/total_params:.1%} of parameters will be trained.")

Total parameters: 11,181,642
Trainable parameters: 5,130
Frozen parameters: 11,176,512

Only 0.0% of parameters will be trained.


## 3. Training Loop
Only the final linear layer has gradients — everything else is frozen.

In [6]:
criterion = nn.CrossEntropyLoss()

# Only pass trainable parameters to the optimizer
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += images.size(0)

    return total_loss / total, correct / total


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += images.size(0)

    return total_loss / total, correct / total


NUM_EPOCHS = 5
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = evaluate(model, val_loader, criterion)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} "
          f"| Train Loss: {train_loss:.4f}  Acc: {train_acc:.2%} "
          f"| Val Loss: {val_loss:.4f}  Acc: {val_acc:.2%}")

Epoch 1/5 | Train Loss: 1.6470  Acc: 52.94% | Val Loss: 1.3982  Acc: 61.75%
Epoch 2/5 | Train Loss: 0.9026  Acc: 80.31% | Val Loss: 0.7246  Acc: 85.00%
Epoch 3/5 | Train Loss: 0.6495  Acc: 87.12% | Val Loss: 0.5647  Acc: 87.75%
Epoch 4/5 | Train Loss: 0.5135  Acc: 88.81% | Val Loss: 0.4916  Acc: 89.00%
Epoch 5/5 | Train Loss: 0.4355  Acc: 91.06% | Val Loss: 0.4361  Acc: 89.75%
